# Spatial Self-Attention Mechanisms vs. Dense Convolutions in Deep Reinforcement Learning for Real-Time Pixel-Based Controls

PPO agents trained from raw pixels on *Super Mario Bros.* levels 1-1, 1-2 and 1-3. Four encoders share one code path:
NatureCNN (baseline), + channel attention (Squeeze-and-Excitation), + spatial self-attention (ours), and an optional parameter-matched pointwise-MLP control.

Run interactively, or headless and resumable on a local machine with `./run_experiments.sh`. Every figure, table, GIF and log is written to `results/`; checkpoints go to `checkpoints/`.

## 1. Setup: install, versions, parameters & output folders

In [ ]:
# Installs whatever is missing. Python ≥ 3.13 (e.g. Kaggle) gets gym-super-mario-bros ≥ 9 (Gymnasium API, NumPy 2);
# older Pythons get the legacy stack (gym 0.25 API, NumPy 1.26). PyTorch is expected to be installed already.
import importlib.util, subprocess, sys

NEW_MARIO_STACK = sys.version_info >= (3, 13)
if any(importlib.util.find_spec(m) is None for m in ("stable_baselines3", "gym_super_mario_bros", "nes_py", "scipy")):
    pkgs = ["stable-baselines3", "pandas", "matplotlib", "pillow", "scipy"]
    pkgs += (["gym-super-mario-bros>=9.1"] if NEW_MARIO_STACK else
             ["gym-super-mario-bros<8", "nes-py<9", "gym==0.25.2", "numpy==1.26.4"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import io, json, os, platform, re, shutil, time, warnings
from collections import deque
from importlib.metadata import version as pkg_version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium
import stable_baselines3 as sb3
from scipy import stats
from PIL import Image
from IPython.display import display, Image as IPyImage

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*out of date.*")
warnings.filterwarnings("ignore", message=".*step API.*")

def _v(pkg):
    try:
        return pkg_version(pkg)
    except PackageNotFoundError:
        return None

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__,
            "gymnasium": gymnasium.__version__, "stable_baselines3": sb3.__version__,
            "gym_super_mario_bros": _v("gym-super-mario-bros"), "nes_py": _v("nes-py"), "gym": _v("gym")}
LEGACY_API = int(VERSIONS["nes_py"].split(".")[0]) < 9          # nes-py < 9: old gym step API
print(" | ".join(f"{k} {v}" for k, v in VERSIONS.items() if v))
print("Mario API:", "legacy gym (4-tuple step)" if LEGACY_API else "gymnasium (5-tuple step)")
print("CPU cores:", os.cpu_count(), "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if LEGACY_API and not np.__version__.startswith("1."):
    raise RuntimeError("nes-py < 9 needs NumPy 1.x: pip install numpy==1.26.4, then restart the kernel")

In [ ]:
# Parameters: papermill overrides these (see run_experiments.sh), e.g. -p TOTAL_TIMESTEPS 2000000 -p TRAIN False
LEVELS = ["1-1", "1-2", "1-3"]
VARIANTS = ["baseline", "channel", "spatial"]     # add "pointwise" for the parameter-matched capacity control
SEEDS = [42, 123, 777]
TOTAL_TIMESTEPS = 1_000_000
N_ENVS = 8                                        # ≈ number of physical CPU cores works best (see throughput check)
TRAIN = True                                      # False: only regenerate results from existing checkpoints and logs
TIME_BUDGET_HOURS = None                          # stop training after N hours (split evenly over remaining runs)
EVAL_EPISODES = 10                                # evaluation episodes per seed and level
GIF_STEPS = 300                                   # decision steps recorded in each attention GIF
TRANSFER = True                                   # zero-shot test: play every trained model on the levels it never saw
REEVALUATE = False                                # False: reuse saved evaluations, throughput, latency and GIFs

In [ ]:
PAPER_TITLE = ("Spatial Self-Attention Mechanisms vs. Dense Convolutions in Deep Reinforcement "
               "Learning for Real-Time Pixel-Based Controls")

CONFIG = {
    "levels": LEVELS, "variants": VARIANTS, "seeds": SEEDS,
    "total_timesteps": TOTAL_TIMESTEPS, "n_envs": N_ENVS, "train": TRAIN,
    "transfer": TRANSFER, "reevaluate": REEVALUATE,
    "rom_version": "v0",            # env id = SuperMarioBros-<level>-<rom_version>: one level, one life per episode
    "frame_skip": 4,
    "frame_stack": 4,
    "reward_scale": 0.1,            # keeps the value loss from swamping the shared CNN; logs keep raw game reward
    "checkpoint_every": 50_000,     # env steps
    "keep_checkpoints": 3,          # newest step-checkpoints kept per run (None = keep all)
    "time_budget_hours": TIME_BUDGET_HOURS,
    "ppo": dict(learning_rate=1e-4, n_steps=512, batch_size=256, n_epochs=4,
                gamma=0.99, ent_coef=0.01, target_kl=None),
    "model": dict(features_dim=512, attn_d_model=64, se_reduction=16),
    # x_pos thresholds for the sample-efficiency metric, set above what a random policy reaches
    # (random-policy median max x ≈ 1400 on 1-1, 800 on 1-2, 300 on 1-3)
    "x_threshold": {"1-1": 2000, "1-2": 1500, "1-3": 1000},
    "smooth_window": 50,            # episodes, for rolling means
    "final_window": 100,            # episodes, for "final performance" metrics
    "eval_episodes": EVAL_EPISODES,
    "eval_deterministic": False,
    "throughput_check": True,
    "gif_steps": GIF_STEPS,
    "gif_fps": 10,
}

ROOT = Path.cwd()
CKPT_DIR = ROOT / "checkpoints"
RESULTS = ROOT / "results"
DIRS = {k: RESULTS / k for k in ["figures", "tables", "gifs", "attention", "logs"]}

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    CONFIG["time_budget_hours"] = CONFIG["time_budget_hours"] or 10.5
    # Resume across Kaggle versions: attach the previous version's output (Add Input → Your Work)
    for sub in ("checkpoints", "results"):
        for prev in sorted(Path("/kaggle/input").glob(f"**/{sub}")):
            if prev.is_dir() and not (ROOT / sub).exists():
                shutil.copytree(prev, ROOT / sub)
                print(f"restored {sub}/ from {prev}")
TRAIN_DEADLINE = time.time() + 3600 * CONFIG["time_budget_hours"] if CONFIG["time_budget_hours"] else None

for d in [CKPT_DIR, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)

LABELS = {"baseline": "NatureCNN (baseline)", "channel": "+ Channel attention (SE)",
          "spatial": "+ Spatial self-attention (ours)", "pointwise": "+ Pointwise MLP (capacity control)",
          "random": "Random policy"}
COLORS = {"baseline": "#4c8bde", "channel": "#3fa36b", "spatial": "#e05c5c", "pointwise": "#9b6bd6",
          "random": "#9a9a9a"}

def run_name(level, variant, seed):
    return f"{level}/{variant}_s{seed}"

def experiment_matrix():
    """Training order: seed-major, so every seed finishes a complete level × variant comparison before the next."""
    return [(lvl, var, s) for s in CONFIG["seeds"] for lvl in CONFIG["levels"] for var in CONFIG["variants"]]

ENV_INFO = {**VERSIONS, "cpu_count": os.cpu_count(),
            "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "platform": "kaggle" if ON_KAGGLE else platform.platform()}
(RESULTS / "config.json").write_text(json.dumps({"title": PAPER_TITLE, "config": CONFIG, "environment": ENV_INFO}, indent=2))
print(f"{len(experiment_matrix())} runs: {len(LEVELS)} levels × {len(VARIANTS)} variants × {len(SEEDS)} seeds "
      f"× {TOTAL_TIMESTEPS:,} steps | results → {RESULTS}")

In [ ]:
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300,
                     "axes.spines.top": False, "axes.spines.right": False})

def save_fig(fig, name, formats=("png", "pdf")):
    """Save to results/figures/<name>.png (300 dpi) and .pdf (vector, for LaTeX)."""
    for ext in formats:
        fig.savefig(DIRS["figures"] / f"{name}.{ext}", bbox_inches="tight")
    print(f"saved → results/figures/{name}.{{{','.join(formats)}}}")

def _fmt(v, floatfmt):
    if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
        return "–"
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (float, np.floating)):
        return format(v, floatfmt)
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    return str(v)

def df_to_markdown(df, floatfmt=",.2f"):
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    rule = "|" + "|".join([":---"] + ["---:"] * (len(df.columns) - 1)) + "|"
    rows = ["| " + " | ".join(_fmt(v, floatfmt) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join([head, rule, *rows])

def df_to_latex(df, floatfmt=",.2f", caption="", label=""):
    def esc(s):
        s = str(s)
        for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"),
                     ("±", r"$\pm$"), ("≥", r"$\geq$"), ("×", r"$\times$"), ("γ", r"$\gamma$"),
                     ("≈", r"$\approx$"), ("−", r"$-$"), ("→", r"$\rightarrow$")]:
            s = s.replace(a, b)
        return s
    body = "\n".join(" & ".join(esc(_fmt(v, floatfmt)) for v in r) + r" \\" for r in df.itertuples(index=False))
    return ("\\begin{table}[t]\n\\centering\n"
            f"\\caption{{{esc(caption)}}}\n\\label{{tab:{label}}}\n"
            f"\\begin{{tabular}}{{l{'r' * (len(df.columns) - 1)}}}\n\\toprule\n"
            + " & ".join(esc(c) for c in df.columns) + " \\\\\n\\midrule\n" + body
            + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")

def save_table(df, name, caption=None, floatfmt=",.2f"):
    """Save to results/tables/<name>.{csv,md,tex} and display it."""
    caption = caption or name
    df.to_csv(DIRS["tables"] / f"{name}.csv", index=False)
    (DIRS["tables"] / f"{name}.md").write_text(f"### {caption}\n\n{df_to_markdown(df, floatfmt)}\n")
    (DIRS["tables"] / f"{name}.tex").write_text(df_to_latex(df, floatfmt, caption, name))
    print(f"saved → results/tables/{name}.{{csv,md,tex}}")
    display(df)

def pm(values, fmt="{:,.0f}"):
    """'mean ± std' over seeds; NaN = not reached in that seed."""
    v = np.asarray(values, dtype=float)
    ok = v[~np.isnan(v)]
    if len(ok) == 0:
        return "not reached"
    s = fmt.format(ok.mean()) if len(ok) == 1 else f"{fmt.format(ok.mean())} ± {fmt.format(ok.std(ddof=1))}"
    return s if len(ok) == len(v) else f"{s} ({len(ok)}/{len(v)} seeds)"

## 2. Explore the raw environment

In [ ]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

ACTION_NAMES = ["+".join(a) for a in SIMPLE_MOVEMENT]

def env_id(level):
    return f"SuperMarioBros-{level}-{CONFIG['rom_version']}"

def make_mario(level, show=False):
    kwargs = {} if LEGACY_API or not show else {"render_mode": "human"}
    return JoypadSpace(gym_super_mario_bros.make(env_id(level), **kwargs), SIMPLE_MOVEMENT)

def mario_reset(env):
    out = env.reset()
    return out[0] if isinstance(out, tuple) else out

def mario_step(env, action):
    """Uniform (frame, reward, terminated, truncated, info) for both the legacy and the gymnasium API."""
    out = env.step(action)
    if len(out) == 4:
        frame, reward, done, info = out
        return frame, reward, done, False, info
    return out

fig, axes = plt.subplots(1, len(LEVELS), figsize=(4 * len(LEVELS), 4), squeeze=False)
for ax, level in zip(axes[0], LEVELS):
    raw = make_mario(level)
    mario_reset(raw)
    for _ in range(60):
        frame, reward, terminated, truncated, info = mario_step(raw, ACTION_NAMES.index("right"))
    raw.close()
    ax.imshow(frame); ax.axis("off"); ax.set_title(f"{env_id(level)}")
print("observation:", frame.shape, "| actions:", ACTION_NAMES)
print("info:", {k: v for k, v in info.items() if k in ("x_pos", "y_pos", "time", "life", "flag_get", "world", "stage")})
save_fig(fig, "env_raw_frames"); plt.show()

## 3. Preprocessing wrapper

In [ ]:
class MarioEnv(gymnasium.Env):
    """One Mario level as a Gymnasium env: frame-skip, 2× downsampling, grayscale, uniform step API."""
    metadata = {"render_modes": []}
    render_mode = None
    OBS_HW = (120, 128)

    def __init__(self, level, skip=4, show=False):
        super().__init__()
        self.env = make_mario(level, show)
        self.skip, self.show = skip, show
        self.action_space = gymnasium.spaces.Discrete(self.env.action_space.n)
        self.observation_space = gymnasium.spaces.Box(0, 255, (*self.OBS_HW, 1), np.uint8)

    @staticmethod
    def _prep(frame):
        small = frame[::2, ::2].astype(np.float32)                    # 240×256 → 120×128
        gray = small[..., 0] * 0.299 + small[..., 1] * 0.587 + small[..., 2] * 0.114
        return gray.astype(np.uint8)[:, :, None]

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return self._prep(mario_reset(self.env)), {}

    def step(self, action):
        total_reward, terminated, truncated, info = 0.0, False, False, {}
        for _ in range(self.skip):
            frame, reward, terminated, truncated, info = mario_step(self.env, int(action))
            total_reward += reward
            if self.show and LEGACY_API:
                self.env.render()
            if terminated or truncated:
                break
        return self._prep(frame), float(total_reward), bool(terminated), bool(truncated), info

    def close(self):
        self.env.close()

e = MarioEnv(LEVELS[0], skip=CONFIG["frame_skip"])
obs, _ = e.reset()
print("obs:", obs.shape, obs.dtype)
e.close()

## 4. Vectorised, frame-stacked training environment

In [ ]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecFrameStack

def make_vec_env(level, n_envs=CONFIG["n_envs"], show=False, monitor=True, reward_scale=1.0):
    def make():
        env = MarioEnv(level, skip=CONFIG["frame_skip"], show=show)
        if monitor:
            env = Monitor(env)                                      # logs the raw episode reward
        if reward_scale != 1.0:
            env = gymnasium.wrappers.TransformReward(env, lambda r: r * reward_scale)
        return env
    venv = DummyVecEnv([make]) if n_envs == 1 else SubprocVecEnv([make] * n_envs, start_method="fork")
    return VecFrameStack(venv, CONFIG["frame_stack"], channels_order="last")

env = make_vec_env(LEVELS[0], n_envs=1)
state = env.reset()
for _ in range(CONFIG["frame_stack"] + 2):                          # fill the stack with distinct frames
    state, *_ = env.step(np.array([ACTION_NAMES.index("right+A")]))
env.close()
print("stacked state:", state.shape, "(n_envs, H, W, frames)")

fig, axes = plt.subplots(1, CONFIG["frame_stack"], figsize=(3.5 * CONFIG["frame_stack"], 3.5))
for i, ax in enumerate(axes):
    ax.imshow(state[0][:, :, i], cmap="gray"); ax.axis("off"); ax.set_title(f"frame t-{CONFIG['frame_stack'] - 1 - i}")
fig.suptitle("Preprocessed observation: 4 stacked 120×128 grayscale frames")
save_fig(fig, "env_preprocessed_framestack"); plt.show()

## 5. Attention blocks: spatial, channel (SE) and capacity control

In [ ]:
class SpatialSelfAttention(nn.Module):
    """Single-head scaled dot-product self-attention over the N = H·W positions of a CNN feature map.

        A   = softmax(Qᵀ K / √d)                (B, N, N)
        out = γ · W_out(V Aᵀ) + x               γ is a learnable gate initialised to 0

    With `store_attn = True` the attention matrix is cached in `last_attn` (CPU) for visualisation;
    it is off during training to avoid a GPU→CPU copy on every minibatch.
    """

    def __init__(self, channels: int, d_model: int = 64):
        super().__init__()
        self.d_model = d_model
        self.q_proj = nn.Conv2d(channels, d_model, kernel_size=1, bias=False)
        self.k_proj = nn.Conv2d(channels, d_model, kernel_size=1, bias=False)
        self.v_proj = nn.Conv2d(channels, d_model, kernel_size=1, bias=False)
        self.out_proj = nn.Conv2d(d_model, channels, kernel_size=1, bias=False)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.store_attn = False
        self.last_attn = None

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        Q = self.q_proj(x).view(B, self.d_model, N)
        K = self.k_proj(x).view(B, self.d_model, N)
        V = self.v_proj(x).view(B, self.d_model, N)
        A = F.softmax(torch.bmm(Q.transpose(1, 2), K) * self.d_model ** -0.5, dim=-1)
        if self.store_attn:
            self.last_attn = A.detach().cpu()
        out = torch.bmm(V, A.transpose(1, 2)).view(B, self.d_model, H, W)
        return self.gamma * self.out_proj(out) + x


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (Hu et al., 2018): reweights channels, no spatial mixing."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(channels, channels // reduction), nn.ReLU(),
                                nn.Linear(channels // reduction, channels), nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x.mean(dim=(2, 3)))
        return x * w[:, :, None, None]


class PointwiseControl(nn.Module):
    """Capacity control: a γ-gated per-position MLP with exactly the spatial-attention parameter count,
    but no mixing across positions. Isolates 'spatial mixing' from 'extra parameters'."""

    def __init__(self, channels: int, d_model: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Conv2d(channels, 2 * d_model, kernel_size=1, bias=False), nn.ReLU(),
                                 nn.Conv2d(2 * d_model, channels, kernel_size=1, bias=False))
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return self.gamma * self.mlp(x) + x

_x = torch.randn(2, 64, 11, 12)
for blk in (SpatialSelfAttention(64), SEBlock(64), PointwiseControl(64)):
    print(f"{type(blk).__name__:22s} out {tuple(blk(_x).shape)} | params {sum(p.numel() for p in blk.parameters()):,}")

## 6. Feature extractor: NatureCNN + optional block

In [ ]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticCnnPolicy

class CNNExtractor(BaseFeaturesExtractor):
    """NatureCNN backbone (Mnih et al., 2015) with an optional block between the last conv and the linear head.
    block='baseline' is exactly SB3's NatureCNN, so every variant shares one code path."""

    def __init__(self, observation_space, features_dim=512, block="spatial", attn_d_model=64, se_reduction=16):
        super().__init__(observation_space, features_dim)
        self.block_name = block
        self.cnn = nn.Sequential(
            nn.Conv2d(observation_space.shape[0], 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        with torch.no_grad():
            _, c, h, w = self.cnn(torch.zeros(1, *observation_space.shape)).shape
        self.feat_chw = (c, h, w)                                    # (64, 11, 12) for 120×128 input
        self.block = {"baseline": lambda: None,
                      "channel": lambda: SEBlock(c, se_reduction),
                      "spatial": lambda: SpatialSelfAttention(c, attn_d_model),
                      "pointwise": lambda: PointwiseControl(c, attn_d_model)}[block]()
        self.linear = nn.Sequential(nn.Flatten(), nn.Linear(c * h * w, features_dim), nn.ReLU())

    def forward(self, obs):
        x = self.cnn(obs)
        if self.block is not None:
            x = self.block(x)
        return self.linear(x)

OBS_SPACE = gymnasium.spaces.Box(0, 255, (CONFIG["frame_stack"], *MarioEnv.OBS_HW), np.uint8)   # channels-first
ACT_SPACE = gymnasium.spaces.Discrete(len(SIMPLE_MOVEMENT))

def policy_kwargs(variant):
    # net_arch=[] keeps SB3's NatureCNN default (heads directly on the 512-d features). Without it SB3 adds
    # 64-64 MLP heads to any custom extractor, so variants would differ by more than the block.
    return dict(net_arch=[], features_extractor_class=CNNExtractor,
                features_extractor_kwargs=dict(block=variant, **CONFIG["model"]))

def build_policy(variant):
    """Untrained actor-critic policy (for parameter counts and latency benchmarks)."""
    return ActorCriticCnnPolicy(OBS_SPACE, ACT_SPACE, lr_schedule=lambda _: 1e-4, **policy_kwargs(variant))

n_params = lambda m: sum(p.numel() for p in m.parameters())
rows = []
for variant in ["baseline", "channel", "spatial", "pointwise"]:
    pol = build_policy(variant)
    blk = pol.features_extractor.block
    rows.append({"Model": LABELS[variant], "Block params": n_params(blk) if blk is not None else 0,
                 "Total policy params": n_params(pol), "Mixes positions": variant == "spatial",
                 "Mixes channels": variant != "baseline"})
arch = pd.DataFrame(rows)
arch.insert(3, "Overhead vs baseline (%)", 100 * (arch["Total policy params"] / arch["Total policy params"].iloc[0] - 1))
save_table(arch, "architecture_params", "Model size (actor-critic policy incl. heads)")

## 7. Logging callbacks

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

class EpisodeCSVLogger(BaseCallback):
    """Appends one row per finished episode to a CSV (survives kernel restarts and resumed runs)
    and prints a one-line status every `print_every` rollouts."""
    FIELDS = ["episode", "timesteps", "reward", "length", "x_pos", "max_x_pos", "flag_get", "unix_time"]

    def __init__(self, csv_path, name="", print_every=10):
        super().__init__()
        self.csv_path, self.name, self.print_every = Path(csv_path), name, print_every
        self.f = None

    def _on_training_start(self):
        new = not self.csv_path.exists() or self.csv_path.stat().st_size == 0
        if not new:
            # after a hard kill the log can run ahead of the last checkpoint: drop rows the model never kept
            df = pd.read_csv(self.csv_path)
            if (df.timesteps > self.model.num_timesteps).any():
                df[df.timesteps <= self.model.num_timesteps].to_csv(self.csv_path, index=False)
                df = df[df.timesteps <= self.model.num_timesteps]
            self.n_episodes = len(df)
        else:
            self.n_episodes = 0
        self.f = open(self.csv_path, "a", buffering=1)
        if new:
            self.f.write(",".join(self.FIELDS) + "\n")
        self.max_x = np.zeros(self.training_env.num_envs)
        self.recent = deque(maxlen=50)
        self.n_rollouts, self.t0, self.ts0 = 0, time.time(), self.model.num_timesteps

    def _on_step(self):
        for i, info in enumerate(self.locals["infos"]):
            x = info.get("x_pos")
            if x is not None:
                self.max_x[i] = max(self.max_x[i], x)
            if "episode" in info:
                self.n_episodes += 1
                flag = int(bool(info.get("flag_get", False)))
                row = [self.n_episodes, self.num_timesteps, f"{info['episode']['r']:.2f}", info["episode"]["l"],
                       "" if x is None else x, int(self.max_x[i]), flag, f"{time.time():.1f}"]
                self.f.write(",".join(map(str, row)) + "\n")
                self.recent.append((info["episode"]["r"], self.max_x[i], flag))
                self.max_x[i] = 0
        return True

    def _on_rollout_end(self):
        self.n_rollouts += 1
        if self.print_every and self.n_rollouts % self.print_every == 0 and self.recent:
            r, x, fl = np.array(self.recent).T
            fps = (self.model.num_timesteps - self.ts0) / max(time.time() - self.t0, 1e-9)
            print(f"[{self.name}] {self.model.num_timesteps:>10,} steps | {fps:5.0f} fps | last {len(r)} eps: "
                  f"reward {r.mean():7.0f} | max x {x.mean():6.0f} | flags {int(fl.sum())}", flush=True)

    def close(self):
        if self.f is not None and not self.f.closed:
            self.f.close()

    def _on_training_end(self):
        self.close()


class GammaCallback(BaseCallback):
    """Records the residual gate γ (spatial / pointwise blocks) into SB3's progress.csv after every rollout."""
    def _on_step(self):
        return True

    def _on_rollout_end(self):
        gamma = getattr(getattr(self.model.policy.features_extractor, "block", None), "gamma", None)
        if gamma is not None:
            self.logger.record("block/gamma", gamma.item())


class DeadlineCallback(BaseCallback):
    """Stops `learn()` cleanly once wall-clock time passes `deadline`."""
    def __init__(self, deadline):
        super().__init__()
        self.deadline = deadline

    def _on_step(self):
        return time.time() < self.deadline

## 8. Training utilities: checkpointing & resume

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.logger import configure
from stable_baselines3.common.utils import set_random_seed

def _prefix(run):
    return run.replace("/", "_")                                     # e.g. 1-1_spatial_s42

def _run_info_path(run):
    return DIRS["logs"] / run / "run_info.json"

def _run_info(run):
    p = _run_info_path(run)
    return json.loads(p.read_text()) if p.exists() else {}

def _step_checkpoints(run):
    return sorted(((int(m.group(1)), p) for p in (CKPT_DIR / run).glob("*.zip")
                   if (m := re.search(r"_(\d+)_steps\.zip$", p.name))), key=lambda c: c[0])

def resolve_checkpoint(run):
    """Most-trained checkpoint of a run: `<run>_final.zip` or the highest `*_<steps>_steps.zip`."""
    cands = _step_checkpoints(run)
    final = CKPT_DIR / run / f"{_prefix(run)}_final.zip"
    if final.exists():
        cands.append((_run_info(run).get("timesteps", 0), final))
    return max(cands, key=lambda c: c[0])[1] if cands else None

def load_run(run, env=None):
    path = resolve_checkpoint(run)
    if path is None:
        raise FileNotFoundError(f"no checkpoint for '{run}' in {CKPT_DIR / run}")
    return PPO.load(path, env=env, device="auto")

def prune_checkpoints(run, keep=CONFIG["keep_checkpoints"]):
    if keep:
        for _, p in _step_checkpoints(run)[:-keep]:
            p.unlink()

def train_run(level, variant, seed, total_timesteps=CONFIG["total_timesteps"]):
    """Train (or resume) one run until `total_timesteps`. Interrupts and time budgets save a checkpoint."""
    run = run_name(level, variant, seed)
    ckpt_dir, log_dir = CKPT_DIR / run, DIRS["logs"] / run
    ckpt_dir.mkdir(parents=True, exist_ok=True); log_dir.mkdir(parents=True, exist_ok=True)
    info = _run_info(run) or {"run": run, "level": level, "variant": variant, "seed": seed, "sessions": 0,
                              "timesteps": 0, "logged_steps": 0, "train_seconds": 0.0}
    if info.get("timesteps", 0) >= total_timesteps:
        print(f"[{run}] already trained to {info['timesteps']:,} steps, skipping")
        return None

    callbacks_extra = []
    if TRAIN_DEADLINE is not None:
        order = [run_name(*r) for r in experiment_matrix()]
        runs_left = [r for r in order[order.index(run):]
                     if r == run or _run_info(r).get("timesteps", 0) < total_timesteps]
        run_deadline = time.time() + (TRAIN_DEADLINE - time.time()) / len(runs_left)
        if run_deadline - time.time() < 60:
            print(f"[{run}] time budget used up, skipping")
            return None
        callbacks_extra.append(DeadlineCallback(run_deadline))
        print(f"[{run}] time budget for this run: {(run_deadline - time.time()) / 3600:.2f} h")

    env = make_vec_env(level, CONFIG["n_envs"], reward_scale=CONFIG["reward_scale"])
    start = resolve_checkpoint(run)
    if start is not None:
        model = PPO.load(start, env=env, device="auto")
        print(f"[{run}] loaded {start.name} ({model.num_timesteps:,} steps)")
    else:
        set_random_seed(seed)
        model = PPO("CnnPolicy", env, policy_kwargs=policy_kwargs(variant), seed=seed,
                    verbose=0, device="auto", **CONFIG["ppo"])
        print(f"[{run}] new model on {model.device}")

    info["sessions"] += 1
    model.set_logger(configure(str(log_dir / f"sb3_session{info['sessions']}"), ["csv"]))
    ep_logger = EpisodeCSVLogger(log_dir / "episodes.csv", name=run)
    callbacks = [CheckpointCallback(save_freq=max(CONFIG["checkpoint_every"] // CONFIG["n_envs"], 1),
                                    save_path=str(ckpt_dir), name_prefix=_prefix(run)),
                 ep_logger, GammaCallback(), *callbacks_extra]
    t0, steps0 = time.time(), model.num_timesteps
    try:
        model.learn(total_timesteps=total_timesteps - model.num_timesteps, callback=callbacks,
                    reset_num_timesteps=False)
        if model.num_timesteps >= total_timesteps:
            model.save(ckpt_dir / f"{_prefix(run)}_final.zip")
            print(f"[{run}] done: {model.num_timesteps:,} steps")
        else:
            model.save(ckpt_dir / f"{_prefix(run)}_{model.num_timesteps}_steps.zip")
            print(f"[{run}] time budget reached at {model.num_timesteps:,} steps; re-run to resume")
    except KeyboardInterrupt:
        model.save(ckpt_dir / f"{_prefix(run)}_{model.num_timesteps}_steps.zip")
        print(f"\n[{run}] interrupted at {model.num_timesteps:,} steps, checkpoint saved; re-run to resume")
        raise
    finally:
        ep_logger.close()
        gamma = getattr(getattr(model.policy.features_extractor, "block", None), "gamma", None)
        info.update(timesteps=int(model.num_timesteps),
                    logged_steps=info["logged_steps"] + int(model.num_timesteps - steps0),
                    train_seconds=round(info["train_seconds"] + time.time() - t0, 1),
                    policy_params=n_params(model.policy),
                    gamma=None if gamma is None else round(gamma.item(), 5))
        info["train_fps"] = round(info["logged_steps"] / max(info["train_seconds"], 1e-9), 1)
        _run_info_path(run).write_text(json.dumps(info, indent=2))
        prune_checkpoints(run)
        env.close()
    return model

## 9. Throughput check

In [ ]:
def throughput_check(level=LEVELS[0], seconds=10):
    """Where does time go? Compares raw emulator speed, parallel env collection and a full PPO iteration."""
    rows, n_envs = [], CONFIG["n_envs"]
    e = MarioEnv(level, CONFIG["frame_skip"]); e.reset()
    n, t0 = 0, time.time()
    while time.time() - t0 < seconds:
        *_, term, trunc, _ = e.step(e.action_space.sample()); n += 1
        if term or trunc:
            e.reset()
    e.close()
    rows.append({"Stage": "1 env, random actions", "Steps/s": n / (time.time() - t0)})

    venv = make_vec_env(level, n_envs)
    venv.reset()
    n, t0 = 0, time.time()
    while time.time() - t0 < seconds:
        venv.step(np.random.randint(ACT_SPACE.n, size=n_envs)); n += n_envs
    rows.append({"Stage": f"{n_envs} envs, random actions", "Steps/s": n / (time.time() - t0)})

    model = PPO("CnnPolicy", venv, policy_kwargs=policy_kwargs("spatial"), verbose=0, device="auto", **CONFIG["ppo"])
    steps = CONFIG["ppo"]["n_steps"] * n_envs
    t0 = time.time()
    model.learn(steps)
    rows.append({"Stage": f"PPO iteration: collect + update on {model.device.type.upper()}",
                 "Steps/s": steps / (time.time() - t0)})
    venv.close()
    df = pd.DataFrame(rows)
    df["Hours per 1M steps"] = 1e6 / df["Steps/s"] / 3600
    return df

if CONFIG["throughput_check"] and (DIRS["tables"] / "throughput.csv").exists() and not REEVALUATE:
    tp = pd.read_csv(DIRS["tables"] / "throughput.csv")
    print("using saved results/tables/throughput.csv (REEVALUATE = True to measure again)")
    display(tp)
elif CONFIG["throughput_check"]:
    tp = throughput_check()
    save_table(tp, "throughput", f"Training throughput ({os.cpu_count()} CPU cores, n_envs={CONFIG['n_envs']})")
if CONFIG["throughput_check"]:
    single, vec, ppo = tp["Steps/s"]
    print(f"\nestimated training time for the full matrix: "
          f"{len(experiment_matrix()) * CONFIG['total_timesteps'] / ppo / 3600:.0f} h")
    if vec < 0.5 * single * min(CONFIG["n_envs"], os.cpu_count()):
        print("→ parallel envs scale poorly: lower N_ENVS towards the number of physical cores")
    if ppo < 0.5 * vec:
        print("→ PPO update/inference dominates: check the GPU is used, or lower n_epochs / raise batch_size")

## 10. Train the experiment matrix

Seed-major order: each seed completes every level × variant before the next seed starts, so stopping early still leaves a complete comparison.

In [ ]:
if CONFIG["train"]:
    for level, variant, seed in experiment_matrix():
        train_run(level, variant, seed)
else:
    print("TRAIN = False: using existing checkpoints and logs")

## 11. Evaluate the trained agents (+ random-policy reference)

In [ ]:
class RandomAgent:
    """Uniform random policy (lower-bound reference)."""
    def predict(self, obs, deterministic=False):
        return np.random.randint(ACT_SPACE.n, size=(obs.shape[0],)), None

def evaluate(agent, level, n_episodes, deterministic=False, max_steps=3000, show=False, seed=0):
    """Play `n_episodes` on one level and return one row per episode."""
    set_random_seed(seed)
    env = make_vec_env(level, 1, show=show, monitor=False)
    rows, obs = [], env.reset()
    for ep in range(1, n_episodes + 1):
        total, max_x = 0.0, 0
        for t in range(1, max_steps + 1):
            action, _ = agent.predict(obs, deterministic=deterministic)
            obs, reward, dones, infos = env.step(action)
            total += float(reward[0])
            max_x = max(max_x, infos[0].get("x_pos", 0))
            if dones[0]:
                break
        else:
            obs = env.reset()
        rows.append({"episode": ep, "reward": total, "steps": t, "max_x_pos": max_x,
                     "flag_get": int(bool(infos[0].get("flag_get", False)))})
    env.close()
    return pd.DataFrame(rows)

EVAL_FILE = DIRS["tables"] / "eval_episodes.csv"
expected = {(l, a, s) for l in LEVELS for a in [*VARIANTS, "random"] for s in SEEDS
            if a == "random" or resolve_checkpoint(run_name(l, a, s)) is not None}
cached = pd.read_csv(EVAL_FILE, dtype={"level": str}) if EVAL_FILE.exists() and not REEVALUATE else None
if cached is not None and expected <= set(map(tuple, cached[["level", "agent", "seed"]].values)):
    evals = cached
    print("using saved results/tables/eval_episodes.csv (REEVALUATE = True to play the episodes again)")
else:
    evals = []
    for level in LEVELS:
        for variant in VARIANTS:
            for seed in SEEDS:
                run = run_name(level, variant, seed)
                if resolve_checkpoint(run) is not None:
                    df = evaluate(load_run(run), level, CONFIG["eval_episodes"], CONFIG["eval_deterministic"], seed=seed)
                    evals.append(df.assign(level=level, agent=variant, seed=seed))
        for seed in SEEDS:
            evals.append(evaluate(RandomAgent(), level, CONFIG["eval_episodes"], seed=seed)
                         .assign(level=level, agent="random", seed=seed))
    evals = pd.concat(evals, ignore_index=True)
    evals.to_csv(EVAL_FILE, index=False)
    print("saved → results/tables/eval_episodes.csv")

RANDOM_REF = evals[evals.agent == "random"].groupby("level").max_x_pos.median().to_dict()

summary = []
for (level, agent), d in evals.groupby(["level", "agent"], sort=False):
    per_seed = d.groupby("seed")
    summary.append({"Level": level, "Agent": LABELS[agent], "Seeds": d.seed.nunique(), "Episodes": len(d),
                    "Max x": pm(per_seed.max_x_pos.mean()), "Flag rate (%)": pm(100 * per_seed.flag_get.mean(), "{:.1f}"),
                    "Reward": pm(per_seed.reward.mean())})
policy_kind = "deterministic" if CONFIG["eval_deterministic"] else "stochastic"
save_table(pd.DataFrame(summary), "eval_summary",
           f"Evaluation: {CONFIG['eval_episodes']} episodes per seed and level ({policy_kind} policy); ± = std over seeds")

agents = [a for a in [*VARIANTS, "random"] if a in set(evals.agent)]
fig, axes = plt.subplots(1, len(LEVELS), figsize=(4.5 * len(LEVELS), 4), squeeze=False)
for ax, level in zip(axes[0], LEVELS):
    d = evals[evals.level == level]
    bp = ax.boxplot([d[d.agent == a].max_x_pos for a in agents], patch_artist=True)
    for patch, a in zip(bp["boxes"], agents):
        patch.set_facecolor(COLORS[a]); patch.set_alpha(0.6)
    ax.set_xticks(range(1, len(agents) + 1), [a for a in agents], rotation=20)
    ax.axhline(CONFIG["x_threshold"][level], color="goldenrod", ls="--", lw=1)
    ax.set_title(f"Level {level}"); ax.set_ylabel("max x_pos per episode")
fig.suptitle("Evaluation: horizontal progress")
fig.tight_layout(); save_fig(fig, "eval_max_x_distribution"); plt.show()

## 12. Learning curves

In [ ]:
def load_episodes(level, variant):
    """Episode logs of all seeds of one level/variant, with a `seed` column."""
    dfs = []
    for seed in SEEDS:
        p = DIRS["logs"] / run_name(level, variant, seed) / "episodes.csv"
        if p.exists() and p.stat().st_size > 0:
            df = pd.read_csv(p)
            if len(df):
                dfs.append(df.assign(seed=seed))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def load_progress(level, variant, seed):
    """SB3 training metrics (progress.csv) across all sessions of a run."""
    files = sorted((DIRS["logs"] / run_name(level, variant, seed)).glob("sb3_session*/progress.csv"))
    dfs = [pd.read_csv(f) for f in files if f.stat().st_size > 0]
    return (pd.concat(dfs, ignore_index=True).sort_values("time/total_timesteps").reset_index(drop=True)
            if dfs else pd.DataFrame())

def curve_on_grid(ep, column, grid, window):
    """Rolling mean of `column` per seed, interpolated onto a shared timestep grid → mean, std, n_seeds."""
    ys = []
    for _, g in ep.groupby("seed"):
        g = g.sort_values("timesteps").assign(_s=lambda d: d[column].rolling(window, min_periods=1).mean())
        g = g.drop_duplicates("timesteps", keep="last")
        if len(g) > 1:
            ys.append(np.interp(grid, g["timesteps"], g["_s"], left=np.nan, right=np.nan))
    if not ys:
        return None, None, 0
    ys = np.array(ys)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(ys, 0), np.nanstd(ys, 0), len(ys)

EPISODES = {(l, v): load_episodes(l, v) for l in LEVELS for v in VARIANTS}
EPISODES = {k: df for k, df in EPISODES.items() if len(df)}
print({f"{l} {v}": f"{len(df):,} episodes" for (l, v), df in EPISODES.items()} or "no training logs yet")

def plot_learning_curves(window=CONFIG["smooth_window"]):
    if not EPISODES:
        return
    panels = [("max_x_pos", "max x_pos", 1), ("flag_get", "flag rate (%)", 100), ("reward", "episode reward", 1)]
    fig, axes = plt.subplots(len(LEVELS), 3, figsize=(16, 4 * len(LEVELS)), squeeze=False)
    for r, level in enumerate(LEVELS):
        runs = {v: df for (l, v), df in EPISODES.items() if l == level}
        if not runs:
            continue
        grid = np.linspace(0, max(df.timesteps.max() for df in runs.values()), 400)
        for c, (col, ylabel, scale) in enumerate(panels):
            ax = axes[r, c]
            for v, ep in runs.items():
                mean, std, n = curve_on_grid(ep, col, grid, window)
                if mean is None:
                    continue
                ax.plot(grid / 1e6, mean * scale, color=COLORS[v], lw=2, label=f"{LABELS[v]} (n={n})")
                if n > 1:
                    ax.fill_between(grid / 1e6, (mean - std) * scale, (mean + std) * scale, color=COLORS[v], alpha=0.15)
            if col == "max_x_pos":
                ax.axhline(CONFIG["x_threshold"][level], color="goldenrod", ls="--", lw=1, label="threshold")
                if level in RANDOM_REF:
                    ax.axhline(RANDOM_REF[level], color=COLORS["random"], ls=":", lw=1.5, label="random policy (median)")
            ax.set_title(f"Level {level}: {ylabel}"); ax.set_xlabel("environment steps (M)")
        axes[r, 0].legend(fontsize=7)
    fig.suptitle(f"{window}-episode rolling mean" + (" (band = ±1 std over seeds)" if len(SEEDS) > 1 else ""), y=1.0)
    fig.tight_layout(); save_fig(fig, "learning_curves"); plt.show()

plot_learning_curves()

In [ ]:
def plot_diagnostics():
    metrics = [("train/entropy_loss", "policy entropy", lambda s: -s, "linear"),
               ("train/approx_kl", "approx. KL", lambda s: s, "log"),
               ("train/explained_variance", "explained variance", lambda s: s, "linear"),
               ("train/value_loss", "value loss", lambda s: s, "log")]
    progress = {(l, v, s): load_progress(l, v, s) for l in LEVELS for v in VARIANTS for s in SEEDS}
    progress = {k: df for k, df in progress.items() if len(df)}
    if not progress:
        print("no SB3 progress logs yet"); return
    fig, axes = plt.subplots(len(LEVELS), 4, figsize=(18, 3.6 * len(LEVELS)), squeeze=False)
    for r, level in enumerate(LEVELS):
        for c, (col, title, f, scale) in enumerate(metrics):
            ax = axes[r, c]
            for (l, v, s), df in progress.items():
                if l == level and col in df:
                    d = df[["time/total_timesteps", col]].dropna()
                    ax.plot(d.iloc[:, 0] / 1e6, f(d[col]), color=COLORS[v], alpha=0.7, lw=1,
                            label=LABELS[v] if s == SEEDS[0] else None)
            ax.set_title(f"Level {level}: {title}"); ax.set_yscale(scale); ax.set_xlabel("environment steps (M)")
        axes[r, 0].legend(fontsize=7)
    fig.tight_layout(); save_fig(fig, "training_diagnostics"); plt.show()

    gam = {k: df for k, df in progress.items() if "block/gamma" in df}
    if gam:
        fig, axes = plt.subplots(1, len(LEVELS), figsize=(5 * len(LEVELS), 3.5), squeeze=False)
        for ax, level in zip(axes[0], LEVELS):
            for (l, v, s), df in gam.items():
                if l == level:
                    d = df[["time/total_timesteps", "block/gamma"]].dropna()
                    ax.plot(d.iloc[:, 0] / 1e6, d.iloc[:, 1], color=COLORS[v],
                            label=LABELS[v] if s == SEEDS[0] else None)
            ax.axhline(0, color="k", lw=0.6)
            ax.set_title(f"Level {level}"); ax.set_xlabel("environment steps (M)"); ax.set_ylabel("γ")
        axes[0, 0].legend(fontsize=7)
        fig.suptitle("Residual gate γ during training (0 = block unused)")
        fig.tight_layout(); save_fig(fig, "block_gamma"); plt.show()

plot_diagnostics()

## 13. Ablation & statistics

In [ ]:
W, LAST = CONFIG["smooth_window"], CONFIG["final_window"]

def run_metrics(ep, budget, threshold):
    """Metrics for one run, restricted to `budget` env steps. AUC = mean of the rolling max-x curve over training."""
    ep = ep[ep.timesteps <= budget].sort_values("timesteps").reset_index(drop=True)
    roll = ep.max_x_pos.rolling(W, min_periods=1).mean()
    grid = np.linspace(0, budget, 200)
    auc = np.interp(grid, ep.timesteps, roll, left=roll.iloc[0]).mean()
    full = ep.max_x_pos.rolling(W, min_periods=W).mean()
    first, sustained = ep[ep.max_x_pos >= threshold], ep[full >= threshold]
    tail = ep.tail(LAST)
    return {"episodes": len(ep),
            "auc_max_x": auc,
            "steps_sustained_x": sustained.timesteps.iloc[0] if len(sustained) else np.nan,
            "steps_first_x": first.timesteps.iloc[0] if len(first) else np.nan,
            "final_max_x": tail.max_x_pos.mean(),
            "final_flag_rate": 100 * tail.flag_get.mean(),
            "final_reward": tail.reward.mean(),
            "best_max_x": ep.max_x_pos.max(),
            "total_flags": int(ep.flag_get.sum())}

per_run = []
for level in LEVELS:
    runs = {k: df for k, df in EPISODES.items() if k[0] == level}
    if not runs:
        continue
    budget = min(g.timesteps.max() for df in runs.values() for _, g in df.groupby("seed"))   # same budget for all
    for (l, v), df in runs.items():
        for seed, g in df.groupby("seed"):
            info = _run_info(run_name(l, v, seed))
            per_run.append({"level": l, "variant": v, "seed": seed, "budget": budget,
                            **run_metrics(g, budget, CONFIG["x_threshold"][l]),
                            "train_fps": info.get("train_fps", np.nan), "gamma": info.get("gamma")})
per_run = pd.DataFrame(per_run)
if len(per_run):
    per_run.to_csv(DIRS["tables"] / "ablation_per_run.csv", index=False)
    print("saved → results/tables/ablation_per_run.csv")

    rows = []
    for (level, v), d in per_run.groupby(["level", "variant"], sort=False):
        T = CONFIG["x_threshold"][level]
        rows.append({"Level": level, "Model": LABELS[v], "Seeds": len(d), "Budget": int(d.budget.iloc[0]),
                     "AUC max x": pm(d.auc_max_x),
                     f"Steps to sustained x ≥ T": pm(d.steps_sustained_x),
                     f"Max x (last {LAST} ep)": pm(d.final_max_x),
                     f"Flag rate % (last {LAST} ep)": pm(d.final_flag_rate, "{:.1f}"),
                     "Train FPS": pm(d.train_fps)})
    save_table(pd.DataFrame(rows), "ablation_summary",
               f"Ablation per level (mean ± std over seeds; T = {CONFIG['x_threshold']}; sustained = {W}-episode mean)")
else:
    print("no training logs yet")

In [ ]:
COMPARISONS = [(a, b) for a, b in [("spatial", "baseline"), ("channel", "baseline"), ("spatial", "channel"),
                                   ("pointwise", "baseline"), ("spatial", "pointwise")]
               if a in VARIANTS and b in VARIANTS]
STAT_METRICS = [("auc_max_x", "AUC max x", False), ("steps_sustained_x", "Steps to sustained x ≥ T", True),
                ("final_max_x", f"Max x (last {LAST} ep)", False), ("final_flag_rate", "Flag rate %", False)]

def bootstrap_diff_ci(a, b, n=10_000, seed=0):
    rng = np.random.default_rng(seed)
    d = rng.choice(a, (n, len(a))).mean(1) - rng.choice(b, (n, len(b))).mean(1)
    return np.percentile(d, [2.5, 97.5])

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    order = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][order]).clip(max=1)
    out[np.flatnonzero(ok)[order]] = adj
    return out

stat_rows = []
if len(per_run):
    for metric, name, lower_better in STAT_METRICS:
        for level in LEVELS:
            for a, b in COMPARISONS:
                da = per_run[(per_run.level == level) & (per_run.variant == a)]
                db = per_run[(per_run.level == level) & (per_run.variant == b)]
                if len(da) < 2 or len(db) < 2:
                    continue
                va, vb = da[metric].to_numpy(float), db[metric].to_numpy(float)
                censored = int(np.isnan(va).sum() + np.isnan(vb).sum())
                if metric == "steps_sustained_x":                    # not reached → censored at the budget
                    va, vb = np.where(np.isnan(va), da.budget, va), np.where(np.isnan(vb), db.budget, vb)
                lo, hi = bootstrap_diff_ci(va, vb)
                p = stats.ttest_ind(va, vb, equal_var=False).pvalue if np.std(va) + np.std(vb) > 0 else np.nan
                stat_rows.append({"Metric": name, "Level": level, "Comparison": f"{a} − {b}",
                                  "Mean A": va.mean(), "Mean B": vb.mean(), "Difference": va.mean() - vb.mean(),
                                  "Relative (%)": 100 * (va.mean() - vb.mean()) / abs(vb.mean()) if vb.mean() else np.nan,
                                  "95% CI low": lo, "95% CI high": hi, "Welch p": p,
                                  "Lower is better": lower_better, "Censored seeds": censored})
if stat_rows:
    st = pd.DataFrame(stat_rows)
    st["Holm p"] = np.nan
    for metric in st.Metric.unique():
        m = st.Metric == metric
        st.loc[m, "Holm p"] = holm(st.loc[m, "Welch p"])
    save_table(st, "ablation_statistics",
               f"Pairwise comparisons (n = {len(SEEDS)} seeds per arm; bootstrap 95% CI of the difference; "
               "Welch t-test, Holm-corrected within each metric)", ",.3g")
    for _, r in st[st.Metric == "AUC max x"].iterrows():
        print(f"{r.Level} {r.Comparison:22s} AUC difference {r.Difference:+8.1f} ({r['Relative (%)']:+.1f}%), "
              f"95% CI [{r['95% CI low']:.0f}, {r['95% CI high']:.0f}], Holm p = {r['Holm p']:.3g}")
else:
    print("statistics need ≥ 2 seeds per variant and level")

In [ ]:
if len(per_run):
    variants = [v for v in VARIANTS if v in set(per_run.variant)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    width = 0.8 / len(variants)
    for ax, (col, title) in zip(axes, [("auc_max_x", "AUC of max x (over training)"),
                                       ("final_max_x", f"max x, last {LAST} episodes")]):
        for i, v in enumerate(variants):
            for j, level in enumerate(LEVELS):
                d = per_run[(per_run.level == level) & (per_run.variant == v)][col]
                if not len(d):
                    continue
                x = j + (i - (len(variants) - 1) / 2) * width
                ax.bar(x, d.mean(), width, yerr=d.std(ddof=1) if len(d) > 1 else None, capsize=3,
                       color=COLORS[v], alpha=0.8, label=LABELS[v] if j == 0 else None)
                ax.scatter(np.full(len(d), x), d, color="k", s=10, zorder=3)
        for j, level in enumerate(LEVELS):
            if level in RANDOM_REF:
                ax.hlines(RANDOM_REF[level], j - 0.45, j + 0.45, color=COLORS["random"], ls=":",
                          label="random policy" if j == 0 else None)
        ax.set_xticks(range(len(LEVELS)), [f"Level {l}" for l in LEVELS]); ax.set_title(title)
    axes[0].legend(fontsize=8)
    fig.suptitle("Per-seed results (dots), mean ± std (bars)")
    fig.tight_layout(); save_fig(fig, "ablation_bars"); plt.show()

## 14. Cross-level transfer (zero-shot)

Every trained model plays the two levels it never saw, with the same protocol as section 11; no training. The diagonal of each matrix is the same-level result from section 11. Scores are also shown relative to the random policy on the test level (1.0 = random play).

In [ ]:
TRANSFER_FILE = DIRS["tables"] / "transfer_episodes.csv"

def transfer_evaluation():
    done = (pd.read_csv(TRANSFER_FILE, dtype={"train_level": str, "test_level": str})
            if TRANSFER_FILE.exists() and not REEVALUATE else pd.DataFrame())
    have = set(map(tuple, done[["train_level", "test_level", "agent", "seed"]].values)) if len(done) else set()
    parts = [done] if len(done) else []
    for train_level in LEVELS:
        for variant in VARIANTS:
            for seed in SEEDS:
                run = run_name(train_level, variant, seed)
                if resolve_checkpoint(run) is None:
                    continue
                model = None
                for test_level in LEVELS:
                    if test_level == train_level or (train_level, test_level, variant, seed) in have:
                        continue
                    if model is None:
                        model = load_run(run)
                    df = evaluate(model, test_level, CONFIG["eval_episodes"], CONFIG["eval_deterministic"], seed=seed)
                    parts.append(df.assign(train_level=train_level, test_level=test_level, agent=variant, seed=seed))
                    pd.concat(parts, ignore_index=True).to_csv(TRANSFER_FILE, index=False)   # saved as it goes
                    print(f"[transfer] {run} -> {test_level}: mean max x {df.max_x_pos.mean():.0f}", flush=True)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if CONFIG["transfer"]:
    off_diag = transfer_evaluation()
    diag = evals[evals.agent != "random"].assign(train_level=lambda d: d.level, test_level=lambda d: d.level)
    TR = pd.concat([off_diag, diag.drop(columns="level")], ignore_index=True)
    TR_SEED = TR.groupby(["train_level", "test_level", "agent", "seed"]).max_x_pos.mean().reset_index()
    RAND_MEAN = evals[evals.agent == "random"].groupby("level").max_x_pos.mean().to_dict()
    TR_SEED["x_random"] = TR_SEED.max_x_pos / TR_SEED.test_level.map(RAND_MEAN)
    TR_SEED.to_csv(DIRS["tables"] / "transfer_per_seed.csv", index=False)
    print("saved → results/tables/transfer_episodes.csv, transfer_per_seed.csv")

    rows = []
    for train_level in LEVELS:
        for v in VARIANTS:
            row = {"Trained on": train_level, "Model": LABELS[v]}
            for test_level in LEVELS:
                d = TR_SEED[(TR_SEED.train_level == train_level) & (TR_SEED.test_level == test_level) & (TR_SEED.agent == v)]
                row[f"Test {test_level}"] = pm(d.max_x_pos) + f" ({d.x_random.mean():.1f}×)" if len(d) else "–"
            rows.append(row)
    rows.append({"Trained on": "–", "Model": LABELS["random"],
                 **{f"Test {l}": f"{RAND_MEAN[l]:,.0f} (1.0×)" for l in LEVELS}})
    save_table(pd.DataFrame(rows), "transfer_matrix",
               "Cross-level transfer: evaluation max x, mean ± std over seeds (multiple of the random policy); "
               "diagonal = trained and tested on the same level")

In [ ]:
if CONFIG["transfer"]:
    vmax = max(1.0, TR_SEED.groupby(["train_level", "test_level", "agent"]).x_random.mean().max())
    fig, axes = plt.subplots(1, len(VARIANTS), figsize=(4.2 * len(VARIANTS), 3.8), squeeze=False)
    for ax, v in zip(axes[0], VARIANTS):
        M = np.full((len(LEVELS), len(LEVELS)), np.nan)
        for i, tr in enumerate(LEVELS):
            for j, te in enumerate(LEVELS):
                d = TR_SEED[(TR_SEED.train_level == tr) & (TR_SEED.test_level == te) & (TR_SEED.agent == v)]
                if len(d):
                    M[i, j] = d.x_random.mean()
        im = ax.imshow(M, cmap="Blues", vmin=0, vmax=vmax)
        for i in range(len(LEVELS)):
            for j in range(len(LEVELS)):
                if not np.isnan(M[i, j]):
                    ax.text(j, i, f"{M[i, j]:.1f}×", ha="center", va="center", fontsize=10,
                            color="white" if M[i, j] > 0.6 * vmax else "#1d1d1b",
                            fontweight="bold" if i == j else "normal")
        ax.set_xticks(range(len(LEVELS)), LEVELS); ax.set_yticks(range(len(LEVELS)), LEVELS)
        ax.set_xlabel("tested on"); ax.set_title(LABELS[v], fontsize=10)
    axes[0, 0].set_ylabel("trained on")
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label="evaluation max x / random policy")
    fig.suptitle("Cross-level transfer (bold diagonal = same level; 1.0× = random play)")
    save_fig(fig, "transfer_matrix"); plt.show()

    stat_rows = []
    for tr in LEVELS:
        for te in LEVELS:
            if tr == te:
                continue
            for a, b in [(x, y) for x, y in [("spatial", "baseline"), ("channel", "baseline"), ("spatial", "channel")]
                         if x in VARIANTS and y in VARIANTS]:
                va = TR_SEED[(TR_SEED.train_level == tr) & (TR_SEED.test_level == te) & (TR_SEED.agent == a)].max_x_pos.to_numpy()
                vb = TR_SEED[(TR_SEED.train_level == tr) & (TR_SEED.test_level == te) & (TR_SEED.agent == b)].max_x_pos.to_numpy()
                if len(va) < 2 or len(vb) < 2:
                    continue
                lo, hi = bootstrap_diff_ci(va, vb)
                p = stats.ttest_ind(va, vb, equal_var=False).pvalue if np.std(va) + np.std(vb) > 0 else np.nan
                stat_rows.append({"Train → test": f"{tr} → {te}", "Comparison": f"{a} − {b}", "Mean A": va.mean(),
                                  "Mean B": vb.mean(), "Difference": va.mean() - vb.mean(),
                                  "95% CI low": lo, "95% CI high": hi, "Welch p": p})
    if stat_rows:
        tst = pd.DataFrame(stat_rows)
        tst["Holm p"] = holm(tst["Welch p"])
        save_table(tst, "transfer_statistics",
                   "Transfer: pairwise differences in evaluation max x on unseen levels (bootstrap 95% CI over seeds; "
                   "Welch t-test, Holm-corrected over all transfer comparisons)", ",.3g")

## 15. Real-time inference latency

In [ ]:
NES_FPS = 60.0988
BUDGET_MS = 1000 * CONFIG["frame_skip"] / NES_FPS          # one decision every `frame_skip` NES frames

@torch.no_grad()
def benchmark_latency(policy, device, batch=1, n_iters=500, warmup=50):
    policy = policy.to(device).eval()
    obs = torch.randint(0, 256, (batch, *OBS_SPACE.shape), dtype=torch.uint8, device=device)
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    for _ in range(warmup):
        policy._predict(obs, deterministic=True)
    sync()
    times = []
    for _ in range(n_iters):
        t = time.perf_counter(); policy._predict(obs, deterministic=True); sync()
        times.append(1000 * (time.perf_counter() - t))
    return np.array(times)

devices = [torch.device("cpu")] + ([torch.device("cuda")] if torch.cuda.is_available() else [])
LAT_FILE = DIRS["tables"] / "inference_latency.csv"
lat_rows = []
for dev in (devices if REEVALUATE or not LAT_FILE.exists() else []):
    for v in VARIANTS:
        ms = benchmark_latency(build_policy(v), dev, n_iters=500 if dev.type == "cuda" else 200)
        lat_rows.append({"Model": LABELS[v], "Device": dev.type.upper(), "Batch": 1,
                         "Mean (ms)": ms.mean(), "p50 (ms)": np.median(ms), "p95 (ms)": np.percentile(ms, 95),
                         "Decisions/s": 1000 / ms.mean(), "Budget (ms)": BUDGET_MS,
                         "Real-time": bool(np.percentile(ms, 95) < BUDGET_MS)})
if lat_rows:
    lat = pd.DataFrame(lat_rows)
else:
    lat = pd.read_csv(LAT_FILE)
    devices = [torch.device(d.lower()) for d in lat.Device.unique()]
    print("using saved results/tables/inference_latency.csv (REEVALUATE = True to measure again)")
save_table(lat, "inference_latency",
           f"Policy forward latency (budget = {CONFIG['frame_skip']} NES frames ≈ {BUDGET_MS:.1f} ms)", ",.3f")

fig, axes = plt.subplots(1, len(devices), figsize=(5.5 * len(devices), 3.5), squeeze=False)
for ax, dev in zip(axes[0], devices):
    d = lat[lat.Device == dev.type.upper()]
    ax.bar(range(len(d)), d["Mean (ms)"], yerr=d["p95 (ms)"] - d["Mean (ms)"],
           color=[COLORS[v] for v in VARIANTS], capsize=4)
    ax.set_xticks(range(len(d)), VARIANTS)
    ax.set_title(dev.type.upper()); ax.set_ylabel("ms per decision (mean, whisker = p95)")
fig.suptitle(f"Inference latency (real-time budget {BUDGET_MS:.1f} ms)")
fig.tight_layout(); save_fig(fig, "inference_latency"); plt.show()

## 16. Attention maps: GIF + paper figure

In [ ]:
def attention_saliency(A, hw, out_hw):
    """(1, N, N) attention → (H_obs, W_obs) map of attention *received* by each position, scaled to [0, 1]."""
    sal = A.mean(dim=1).view(1, 1, *hw).float()
    sal = F.interpolate(sal, size=out_hw, mode="bilinear", align_corners=False).squeeze().numpy()
    lo, hi = sal.min(), sal.max()
    return ((sal - lo) / (hi - lo) if hi > lo else np.zeros_like(sal)).astype(np.float32)

def attention_entropies(A):
    """Normalised entropies in [0, 1] (1 = uniform): mean over query rows, and of the received distribution."""
    a = A[0].double().clamp_min(1e-12)
    log_n = np.log(a.shape[-1])
    row = float((-(a * a.log()).sum(-1)).mean() / log_n)
    p = a.mean(0)
    return row, float(-(p * p.log()).sum() / log_n)

def record_attention(run, level, n_steps, deterministic=False, seed=0):
    """Roll out a trained spatial-attention model; save frames, saliency maps and per-step stats."""
    model = load_run(run)
    ext = model.policy.features_extractor
    attn = getattr(ext, "block", None)
    # compare by name: SB3 pickles notebook-defined classes by value, so a model loaded in a new session
    # carries its own copy of the class and isinstance() would fail
    if type(attn).__name__ != "SpatialSelfAttention":
        raise TypeError(f"'{run}' has no spatial attention block")
    model.policy.set_training_mode(False)
    attn.store_attn = True
    set_random_seed(seed)
    env = make_vec_env(level, 1, monitor=False)
    obs = env.reset()
    frames, sals, rows = [], [], []
    episode, ret = 1, 0.0
    try:
        for step in range(n_steps):
            action, _ = model.predict(obs, deterministic=deterministic)
            A = attn.last_attn
            frames.append(obs[0, :, :, -1].copy())
            sals.append(attention_saliency(A, ext.feat_chw[1:], obs.shape[1:3]))
            row_h, recv_h = attention_entropies(A)
            act = ACTION_NAMES[int(action[0])]
            obs, rew, dones, infos = env.step(action)
            rows.append({"step": step, "episode": episode, "action": act, "return_before": ret,
                         "x_pos": infos[0].get("x_pos"), "row_entropy": row_h, "received_entropy": recv_h,
                         "done": bool(dones[0])})
            ret += float(rew[0])
            if dones[0]:
                episode, ret = episode + 1, 0.0
    finally:
        attn.store_attn = False
        env.close()
    tag = _prefix(run)
    frames, sals, stats_df = np.array(frames, np.uint8), np.array(sals, np.float16), pd.DataFrame(rows)
    np.savez_compressed(DIRS["attention"] / f"attention_maps_{tag}.npz", frames=frames, saliency=sals,
                        actions=stats_df.action.to_numpy(), gamma=attn.gamma.item())
    stats_df.to_csv(DIRS["attention"] / f"attention_stats_{tag}.csv", index=False)
    print(f"saved → results/attention/attention_maps_{tag}.npz, attention_stats_{tag}.csv  (γ = {attn.gamma.item():.4f})")
    return frames, sals, stats_df

In [ ]:
def overlay(gray, sal, cmap, alpha):
    rgb = np.repeat(gray[..., None], 3, axis=2)
    return np.clip((1 - alpha) * rgb + alpha * matplotlib.colormaps[cmap](sal)[..., :3], 0, 1)

def render_attention_frame(gray, sal, action, step, ret, cmap="plasma", alpha=0.55, dpi=90):
    """Three panels (frame | attention | overlay) rendered to a palettised PIL image."""
    cm = matplotlib.colormaps[cmap]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), dpi=dpi)
    fig.patch.set_facecolor("#111111")
    for ax, img, title in [(axes[0], np.repeat(gray[..., None], 3, 2), "Game frame"),
                           (axes[1], cm(sal)[..., :3], "Attention map"),
                           (axes[2], overlay(gray, sal, cmap, alpha), "Overlay")]:
        ax.imshow(img, vmin=0, vmax=1); ax.set_title(title, color="white", fontsize=9, pad=3); ax.axis("off")
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cm, norm=plt.Normalize(0, 1)), ax=axes[1], fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors="white", labelsize=7); cb.set_label("attention", color="white", fontsize=7)
    fig.text(0.5, 0.01, f"step {step:4d}  |  action {action:<12s}  |  return {ret:+.0f}",
             ha="center", color="#cccccc", fontsize=8)
    fig.tight_layout(rect=[0, 0.04, 1, 1])
    fig.canvas.draw()
    img = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())
    plt.close(fig)
    return img.convert("P", palette=Image.Palette.ADAPTIVE)

def save_attention_gif(tag, frames, sals, stats_df, fps=CONFIG["gif_fps"], cmap="plasma", alpha=0.55, dpi=90):
    imgs = [render_attention_frame(f / 255.0, s.astype(np.float32), r.action, r.step, r.return_before, cmap, alpha, dpi)
            for f, s, r in zip(frames, sals, stats_df.itertuples())]
    path = DIRS["gifs"] / f"attention_{tag}.gif"
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=int(1000 / fps), loop=0, optimize=True)
    print(f"saved → results/gifs/{path.name} ({len(imgs)} frames, {path.stat().st_size / 2**20:.1f} MB)")
    return path

def pick_frames(stats_df, k=6, mode="spread"):
    """'spread' = evenly spaced; 'focused' = lowest received-attention entropy, at least n/(2k) steps apart."""
    if mode == "spread":
        return np.linspace(0, len(stats_df) - 1, k).astype(int)
    chosen, gap = [], max(len(stats_df) // (2 * k), 1)
    for i in stats_df.received_entropy.argsort():
        if all(abs(i - j) >= gap for j in chosen):
            chosen.append(int(i))
        if len(chosen) == k:
            break
    return np.sort(chosen)

def plot_attention_grid(tag, frames, sals, stats_df, k=6, mode="spread", cmap="plasma", alpha=0.55):
    idx = pick_frames(stats_df, k, mode)
    fig, axes = plt.subplots(2, len(idx), figsize=(2.6 * len(idx), 5.2), squeeze=False)
    for c, i in enumerate(idx):
        g = frames[i] / 255.0
        axes[0, c].imshow(g, cmap="gray", vmin=0, vmax=1)
        axes[1, c].imshow(overlay(g, sals[i].astype(np.float32), cmap, alpha))
        axes[0, c].set_title(f"t={stats_df.step[i]} · {stats_df.action[i]}", fontsize=8)
        for ax in axes[:, c]:
            ax.axis("off")
    axes[0, 0].text(-0.08, 0.5, "frame", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
    axes[1, 0].text(-0.08, 0.5, "attention", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
    fig.tight_layout(); save_fig(fig, f"attention_grid_{mode}_{tag}"); plt.show()

def plot_attention_trace(tag, stats_df):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(stats_df.step, stats_df.received_entropy, color=COLORS["spatial"], label="received-attention entropy")
    ax.plot(stats_df.step, stats_df.row_entropy, color=COLORS["spatial"], ls=":", label="mean row entropy")
    ax.set_xlabel("decision step"); ax.set_ylabel("normalised entropy (1 = uniform)")
    ax2 = ax.twinx()
    ax2.plot(stats_df.step, stats_df.x_pos, color="#555555", alpha=0.6, label="x_pos"); ax2.set_ylabel("x_pos")
    for s in stats_df.step[stats_df.done]:
        ax.axvline(s, color="k", lw=0.6, ls="--")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="lower left")
    ax.set_title("Attention focus along the rollout (dashed = episode end)")
    fig.tight_layout(); save_fig(fig, f"attention_entropy_{tag}"); plt.show()

In [ ]:
last_gif = None
if "spatial" in VARIANTS:
    for level in LEVELS:
        run = run_name(level, "spatial", SEEDS[0])
        if resolve_checkpoint(run) is None:
            print(f"no checkpoint for {run} yet"); continue
        tag = _prefix(run)
        if (DIRS["gifs"] / f"attention_{tag}.gif").exists() and not REEVALUATE:
            print(f"using saved results/gifs/attention_{tag}.gif"); last_gif = DIRS["gifs"] / f"attention_{tag}.gif"; continue
        frames, sals, attn_stats = record_attention(run, level, CONFIG["gif_steps"])
        last_gif = save_attention_gif(tag, frames, sals, attn_stats)
        plot_attention_grid(tag, frames, sals, attn_stats, mode="spread")
        plot_attention_grid(tag, frames, sals, attn_stats, mode="focused")
        plot_attention_trace(tag, attn_stats)
if last_gif is not None:
    display(IPyImage(filename=str(last_gif)))

## 17. Watch the agent play

In [ ]:
WATCH = False   # True opens an emulator window
if WATCH:
    print(evaluate(load_run(run_name(LEVELS[0], "spatial", SEEDS[0])), LEVELS[0], n_episodes=3, show=True))

## 18. Results summary

In [ ]:
TABLE_ORDER = ["architecture_params", "throughput", "ablation_summary", "ablation_statistics",
               "eval_summary", "transfer_matrix", "transfer_statistics", "inference_latency"]
lines = [f"# Results: {PAPER_TITLE}", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} · levels {LEVELS} · variants {VARIANTS} · seeds {SEEDS} · "
         f"{CONFIG['total_timesteps']:,} target steps per run · config in `config.json`", ""]
for name in TABLE_ORDER:
    p = DIRS["tables"] / f"{name}.md"
    if p.exists():
        lines += [p.read_text(), ""]
lines += ["## Figures", ""] + [f"![{p.stem}](figures/{p.name})" for p in sorted(DIRS["figures"].glob("*.png"))]
lines += ["", "## Animations", ""] + [f"![{p.stem}](gifs/{p.name})" for p in sorted(DIRS["gifs"].glob("*.gif"))]
(RESULTS / "summary.md").write_text("\n".join(lines) + "\n")
print("saved → results/summary.md\n")

for p in sorted(RESULTS.rglob("*")):
    if p.is_file() and "logs" not in p.parts:
        print(f"{p.stat().st_size / 1024:9.1f} KB  {p.relative_to(ROOT)}")